# When calibration ages: temporal drift and economic decisions

Our earlier calibration notebook assumes exchangeable calibration and test observations. Here we replace random evaluation with a chronological deployment stream and inspect fixed versus rolling calibration, with and without periodic refitting.

We separate four controlled regimes: stable data, covariate shift, inflated observation noise, and a genuine change in the conditional response. Each regime shares the same random innovations before the specified midpoint. We evaluate predictive coverage and economic regret before and after the change.

Under a stable IID stream, initial split conformal has a marginal exchangeability argument. This is not a time-uniform or conditional guarantee. Under drift, the guarantee need not hold. Rolling residual quantiles and refitted models are practical heuristics, not guaranteed conformal methods.

In [ ]:
from __future__ import annotations
import os
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from temporal_drift import run_temporal_experiment, aggregate_temporal_coverage, aggregate_temporal_decisions

REPEATS = int(os.environ.get("DRIFT_REPEATS", "8"))
SMOKE = os.environ.get("DRIFT_SMOKE", "0") == "1"
if REPEATS < 1:
    raise ValueError("DRIFT_REPEATS must be positive")
REGIMES = ("stable", "concept_shift") if SMOKE else (
    "stable", "covariate_shift", "noise_shift", "concept_shift"
)
SCENARIOS = ("low_variance_signal",) if SMOKE else (
    "low_variance_signal", "high_variance_signal"
)

## Prequential evaluation

At time \(t\), each method must first predict \(Y_t\), construct its interval and choose its action. Only after these outputs have been saved may it observe \(Y_t\) and update rolling residuals. The periodically refitted model likewise uses only records indexed before \(t\).

Methods:

- Frozen/static: train once, calibrate once, never update.
- Frozen/rolling: keep model fixed, update the residual quantile using strictly past outcomes.
- Refit/rolling: periodically refit on the most recent labelled history, with rolling past prequential residual scores.

Refitting makes scores non-identically distributed even when observations are stable. Therefore neither rolling method inherits the original split-conformal coverage theorem.

In [ ]:
intervals, decisions = run_temporal_experiment(
    repeats=REPEATS,
    n_fit=80 if SMOKE else 160,
    n_cal=30 if SMOKE else 80,
    n_deploy=40 if SMOKE else 120,
    rolling_window=20 if SMOKE else 60,
    refit_every=10 if SMOKE else 20,
    refit_window=64 if SMOKE else 100,
    scenarios=SCENARIOS,
    regimes=REGIMES,
)
assert intervals["covered"].between(0, 1).all()
assert np.isfinite(decisions["regret"]).all()
coverage = aggregate_temporal_coverage(intervals, bootstrap_draws=200)
economic = aggregate_temporal_decisions(decisions)
display(coverage.round(3))
display(economic.head(15).round(3))

## What breaks and what adapts?

Compare nominal 90% interval coverage, width, intervention frequency and expected regret before versus after the known breakpoint. Model recalibration can help against noise changes, but it cannot automatically repair an incorrect conditional relationship. Refit strategies may adapt to concept shift if there are enough post-shift outcomes.

The economic oracle knows the synthetic conditional mean only for evaluation. These are *predictive decision* experiments, not causal policy analyses. The low-bound rule uses an interval for noisy \(Y\), which has no formal guarantee of profitable action.

In [ ]:
case = coverage.loc[
    (coverage["scenario"] == "low_variance_signal")
    & (coverage["regime"] == "concept_shift")
]
fig, ax = plt.subplots(figsize=(8, 4))
for name, sub in case.groupby("policy"):
    sub = sub.set_index("phase").reindex(["before_shift", "after_shift"])
    ax.plot(["Before", "After"], sub["mean_coverage"], marker="o", label=name)
ax.axhline(0.9, linestyle="--", color="grey", label="Nominal 90%")
ax.set(ylabel="Observed interval coverage", ylim=(0, 1.02),
       title="Concept shift: coverage before and after breakpoint")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
focus = economic.loc[
    (economic["scenario"] == "low_variance_signal")
    & (economic["regime"] == "concept_shift")
    & (economic["cost"] == 0.6)
    & (economic["rule"] == "Point threshold")
]
display(focus.sort_values(["phase", "mean_regret"]).round(3))

## Interpretation and extensions

The empirical coverage intervals should resample entire independently generated streams, never individual time steps as if independent. Eight streams are exploratory, not enough for accurate uncertainty claims.

This simulation uses independent innovations with an abrupt change point. It does not yet test autocorrelation, overlapping observations, delayed outcomes, seasonal drift, unknown change times or adaptive intervals with proved dynamic-regret bounds.

Next: introduce autocorrelation, delayed label feedback and real-world data under provenance and leakage checks before making stronger economic conclusions.